In [ ]:
!pip install -q transformers==4.44.2 accelerate>=1.0 sentence-transformers

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

NOME_MODELLO = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(NOME_MODELLO)
model = AutoModelForCausalLM.from_pretrained(
    NOME_MODELLO,
    torch_dtype=torch.float16,
    device_map="auto",
)

In [ ]:
import torch.nn.functional as F
class ModelloPerplexity:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer
        self.device = model.device

    @torch.no_grad()
    def logprob_testo(self, testo_nodo, contesto=""):
        if contesto:
            prompt = contesto + "\n\n" + testo_nodo
            n_ctx_tok = self.tokenizer(contesto + "\n\n", return_tensors="pt").input_ids.shape[1]
        else:
            prompt = testo_nodo
            n_ctx_tok = 1

        ids = self.tokenizer(prompt, return_tensors="pt").input_ids.to(self.device)
        logits = self.model(ids).logits

        logprobs = F.log_softmax(logits[0, :-1], dim=-1)
        target = ids[0, 1:]
        tok_logprob = logprobs[torch.arange(target.shape[0]), target]

        nodo_logprob = tok_logprob[n_ctx_tok - 1:]
        somma = nodo_logprob.sum().item()
        n_token = nodo_logprob.shape[0]
        return somma, n_token

    def perplexity_testo(self, testo_nodo, contesto=""):
        somma, n_token = self.logprob_testo(testo_nodo, contesto)
        if n_token == 0:
            return float("inf")
        return float(torch.exp(torch.tensor(-somma / n_token)))

In [ ]:
m = ModelloPerplexity(model, tokenizer)

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx

BASE = "/kaggle/input/datasets/angelopaldino02/datasetcora"

# ordine nodi da cora.content
content = pd.read_csv(f"{BASE}/cora.content", sep="\t", header=None)
paper_ids = content.iloc[:, 0].astype(str).values
n_nodi = len(paper_ids)
id_to_index = {pid: i for i, pid in enumerate(paper_ids)}

# archi da cora.cites
cites = pd.read_csv(f"{BASE}/cora.cites", sep="\t", header=None, names=["cited", "citing"], dtype=str)
G = nx.Graph()
G.add_nodes_from(range(n_nodi))
for cited, citing in zip(cites["cited"], cites["citing"]):
    if cited in id_to_index and citing in id_to_index:
        G.add_edge(id_to_index[citing], id_to_index[cited])

# testo + etichette dal CSV combinato, ordinato per id
tape = pd.read_csv(f"{BASE}/combined.csv").sort_values("id").reset_index(drop=True)

testi = []
for i in range(n_nodi):
    t = tape.loc[i, "T"]; a = tape.loc[i, "A"]
    t = "" if pd.isna(t) else str(t).strip()
    a = "" if pd.isna(a) else str(a).strip()
    if t and a: testi.append(f"{t}. {a}")
    elif t: testi.append(t)
    elif a: testi.append(a)
    else: testi.append("")

etichette = tape["label"].values.astype(int)
print(f"Nodi: {n_nodi}, Archi: {G.number_of_edges()}, Classi: {len(set(etichette))}")

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")

testi_per_embedding = [t[:1000] if t else "" for t in testi]
embeddings = embedder.encode(testi_per_embedding, convert_to_numpy=True, show_progress_bar=True, batch_size=64)
print("Shape embeddings:", embeddings.shape)  # (2708, 384)

In [ ]:
!pip install infomap

In [ ]:
import infomap
import numpy as np


class ValutatoreCodelength:
    """
    Valuta L(C) (description length della map equation) di una partizione,
    usando la libreria infomap ufficiale.

    L'oggetto Infomap viene costruito una volta sola con gli archi del grafo;
    per ogni valutazione si assegna la partizione candidata e si esegue
    run(no_infomap=True), che calcola L(C) senza ottimizzare.
    """

    def __init__(self, grafo):
        self.archi = [(int(u), int(v)) for u, v in grafo.edges()]
        self.nodi = sorted(int(n) for n in grafo.nodes())
        self._im = infomap.Infomap(silent=True, num_trials=1)
        self._im.add_links(self.archi)

    def codelength(self, partizione):
        self._im.initial_partition = {n: int(partizione[n]) + 1 for n in self.nodi}
        self._im.run(no_infomap=True)
        return float(self._im.codelength)

    def codelength_con_mossa(self, partizione, nodo, nuova_community):
        """L(C) della partizione ottenuta spostando `nodo` in `nuova_community`."""
        vecchia = partizione[nodo]
        partizione[nodo] = nuova_community
        lc = self.codelength(partizione)
        partizione[nodo] = vecchia   # ripristino
        return lc

In [ ]:
import numpy as np
from tqdm import tqdm


class ScorerPerplexity:
    def __init__(self, modello, testi, embeddings, max_nodi_contesto=20, max_char_testo=300):
        self.modello = modello
        self.testi = testi
        self.embeddings = embeddings
        self.max_nodi_contesto = max_nodi_contesto
        self.max_char_testo = max_char_testo

    def _costruisci_contesto(self, nodo, membri_community):
        altri = [m for m in membri_community if m != nodo]
        if not altri:
            return ""
        # centroide semantico della community (esclusi il nodo stesso)
        centroide = self.embeddings[altri].mean(axis=0)
        # scelgo i nodi piu' vicini al centroide (cosine similarity)
        emb_altri = self.embeddings[altri]
        sim = emb_altri @ centroide / (
            np.linalg.norm(emb_altri, axis=1) * np.linalg.norm(centroide) + 1e-8
        )
        k = min(self.max_nodi_contesto, len(altri))
        idx_top = np.argsort(sim)[::-1][:k]
        selezionati = [altri[i] for i in idx_top]
        pezzi = [self.testi[m][:self.max_char_testo] for m in selezionati if self.testi[m]]
        return "\n\n".join(pezzi)

    def perplexity_nodo(self, nodo, membri_community):
        testo = self.testi[nodo][:self.max_char_testo]
        if not testo:
            return None
        contesto = self._costruisci_contesto(nodo, membri_community)
        return self.modello.perplexity_testo(testo, contesto)

    def ppl_partizione(self, partizione, mostra_avanzamento=True, descrizione="PPL(C)"):
        community = {}
        for nodo, c in enumerate(partizione):
            community.setdefault(c, []).append(nodo)

        ppl_per_nodo = {}
        iteratore = range(len(partizione))
        if mostra_avanzamento:
            iteratore = tqdm(iteratore, desc=descrizione, unit="nodo")

        for nodo in iteratore:
            c = partizione[nodo]
            ppl = self.perplexity_nodo(nodo, community[c])
            if ppl is not None:
                ppl_per_nodo[nodo] = ppl

        if not ppl_per_nodo:
            return float("inf"), {}
        media = float(np.mean(list(ppl_per_nodo.values())))
        return media, ppl_per_nodo

In [ ]:
import numpy as np

def _seleziona_contesto_da_sorgente(self, nodo_target, membri_sorgente):
    """
    Costruisce il contesto per predire nodo_target usando SOLO i nodi di
    membri_sorgente (che possono appartenere a un'altra community).
    Sceglie i piu' vicini semanticamente al nodo target.
    """
    altri = [m for m in membri_sorgente if m != nodo_target]
    if not altri:
        return ""
    emb_target = self.embeddings[nodo_target]
    emb_altri = self.embeddings[altri]
    sim = emb_altri @ emb_target / (
        np.linalg.norm(emb_altri, axis=1) * np.linalg.norm(emb_target) + 1e-8
    )
    k = min(self.max_nodi_contesto, len(altri))
    idx_top = np.argsort(sim)[::-1][:k]
    selezionati = [altri[i] for i in idx_top]
    pezzi = [self.testi[m][:self.max_char_testo] for m in selezionati if self.testi[m]]
    return "\n\n".join(pezzi)

def perplexity_nodo_con_sorgente(self, nodo_target, membri_sorgente):
    """
    Perplessita' del testo di nodo_target predetto col contesto di membri_sorgente.
    Se sorgente == community del nodo -> perplessita' interna.
    Se sorgente == altra community    -> perplessita' incrociata.
    """
    testo = self.testi[nodo_target][:self.max_char_testo]
    if not testo:
        return None
    contesto = self._seleziona_contesto_da_sorgente(nodo_target, membri_sorgente)
    return self.modello.perplexity_testo(testo, contesto)

ScorerPerplexity._seleziona_contesto_da_sorgente = _seleziona_contesto_da_sorgente
ScorerPerplexity.perplexity_nodo_con_sorgente = perplexity_nodo_con_sorgente

In [ ]:
scorer = ScorerPerplexity(m, testi, embeddings)

In [ ]:
import numpy as np
from tqdm import tqdm


def diagnostica_normalizzazione(scorer, valutatore, grafo, partizione,
                                n_campione=100, seed=42):
    """
    Campiona n_campione mosse candidate reali (nodo -> community di un vicino)
    e misura le variazioni dei due termini:
      - delta_sem  = log2(PPL del nodo nella nuova community) - log2(PPL nella attuale)
      - delta_lc   = L(C) dopo la mossa - L(C) prima

    Restituisce le deviazioni standard, da usare come fattori di normalizzazione,
    piu' le statistiche complete per ispezione.
    """
    rng = np.random.default_rng(seed)
    partizione = np.array(partizione).copy()
    n_nodi = len(partizione)

    # mappa community -> membri (ricalcolata una volta)
    membri_com = {}
    for nodo, c in enumerate(partizione):
        membri_com.setdefault(c, []).append(nodo)

    lc_base = valutatore.codelength(partizione)

    delta_sem, delta_lc = [], []
    tentativi = 0
    barra = tqdm(total=n_campione, desc="diagnostica scale", unit="mossa")

    while len(delta_sem) < n_campione and tentativi < n_campione * 10:
        tentativi += 1
        nodo = int(rng.integers(0, n_nodi))
        vicini = list(grafo.neighbors(nodo))
        if not vicini:
            continue
        com_attuale = partizione[nodo]
        # community candidate = quelle dei vicini, diverse dall'attuale
        candidate = {partizione[v] for v in vicini} - {com_attuale}
        if not candidate:
            continue
        com_nuova = int(rng.choice(list(candidate)))

        # --- termine semantico ---
        ppl_attuale = scorer.perplexity_nodo(nodo, membri_com[com_attuale])
        membri_nuova = membri_com[com_nuova] + [nodo]
        ppl_nuova = scorer.perplexity_nodo(nodo, membri_nuova)
        if ppl_attuale is None or ppl_nuova is None or ppl_attuale <= 0 or ppl_nuova <= 0:
            continue
        d_sem = np.log2(ppl_nuova) - np.log2(ppl_attuale)

        # --- termine strutturale ---
        lc_nuova = valutatore.codelength_con_mossa(partizione, nodo, com_nuova)
        d_lc = lc_nuova - lc_base

        delta_sem.append(d_sem)
        delta_lc.append(d_lc)
        barra.update(1)
    barra.close()

    delta_sem = np.array(delta_sem)
    delta_lc = np.array(delta_lc)

    sigma_sem = float(np.std(delta_sem)) or 1e-8
    sigma_lc = float(np.std(delta_lc)) or 1e-8

    print("\n=== Diagnostica delle scale ===")
    print(f"mosse campionate: {len(delta_sem)}")
    print(f"delta_sem : media {delta_sem.mean():+.4f} | std {sigma_sem:.4f} | "
          f"range [{delta_sem.min():+.4f}, {delta_sem.max():+.4f}]")
    print(f"delta_L(C): media {delta_lc.mean():+.6f} | std {sigma_lc:.6f} | "
          f"range [{delta_lc.min():+.6f}, {delta_lc.max():+.6f}]")
    print(f"rapporto delle scale (sigma_sem / sigma_lc): {sigma_sem/sigma_lc:.1f}x")
    print("I due termini vanno divisi per le rispettive sigma prima di combinarli.")

    return {
        "sigma_sem": sigma_sem,
        "sigma_lc": sigma_lc,
        "delta_sem": delta_sem.tolist(),
        "delta_lc": delta_lc.tolist(),
    }

In [ ]:
partizione_infomap = np.load(f"/kaggle/input/datasets/angelopaldino02/datasetcora/baseline_infomap.npy")
val = ValutatoreCodelength(G)
scale = diagnostica_normalizzazione(scorer, val, G, partizione_infomap, n_campione=100)

In [ ]:
import numpy as np
from tqdm import tqdm


class LocalMovingIbrido:
    """
    Local-moving che minimizza L_ibrida(C) = alpha*H_sem(C) + (1-alpha)*L_strut(C),
    dove H_sem = log2(PPL) e' l'entropia semantica e L_strut la description length
    della map equation (calcolata con la libreria infomap ufficiale).

    Per ogni mossa candidata si valutano:
      delta_sem = log2(PPL nella candidata) - log2(PPL nell'attuale)
      delta_lc  = L(C) dopo la mossa - L(C) prima
    e si combinano, normalizzati per le rispettive sigma:
      guadagno = alpha*(-delta_sem/sigma_sem) + (1-alpha)*(-delta_lc/sigma_lc)
    Si sposta il nodo nella candidata con guadagno massimo, se > soglia.
    """

    def __init__(self, scorer, valutatore, grafo, partizione, alpha,
                 sigma_sem, sigma_lc, soglia=0.0, max_iter=4,
                 dir_salvataggio="/kaggle/working", seed=42):
        self.scorer = scorer
        self.valutatore = valutatore
        self.grafo = grafo
        self.partizione = np.array(partizione).copy()
        self.alpha = alpha
        self.sigma_sem = max(sigma_sem, 1e-8)
        self.sigma_lc = max(sigma_lc, 1e-8)
        self.soglia = soglia
        self.max_iter = max_iter
        self.dir_salvataggio = dir_salvataggio
        self.rng = np.random.default_rng(seed)
        self.membri = self._mappa_membri()

    def _mappa_membri(self):
        m = {}
        for nodo, c in enumerate(self.partizione):
            m.setdefault(int(c), []).append(nodo)
        return m

    def _sposta(self, nodo, nuova):
        vecchia = int(self.partizione[nodo])
        if vecchia == nuova:
            return
        self.membri[vecchia].remove(nodo)
        if not self.membri[vecchia]:
            del self.membri[vecchia]
        self.membri.setdefault(nuova, []).append(nodo)
        self.partizione[nodo] = nuova

    def _entropia_sem(self, nodo, membri_community):
        ppl = self.scorer.perplexity_nodo(nodo, membri_community)
        if ppl is None or ppl <= 0:
            return None
        return float(np.log2(ppl))

    def _valuta_mossa(self, nodo, com_candidata, h_attuale, lc_base):
        # termine semantico: il contesto e' fatto dai membri attuali della candidata
        membri_cand = self.membri[com_candidata]
        h_cand = self._entropia_sem(nodo, membri_cand + [nodo])
        if h_cand is None or h_attuale is None:
            return None
        delta_sem = h_cand - h_attuale

        # termine strutturale
        lc_cand = self.valutatore.codelength_con_mossa(self.partizione, nodo, com_candidata)
        delta_lc = lc_cand - lc_base

        guadagno = (self.alpha * (-delta_sem / self.sigma_sem)
                    + (1 - self.alpha) * (-delta_lc / self.sigma_lc))
        return guadagno, delta_sem, delta_lc

    def esegui(self):
        storico = []
        for it in range(1, self.max_iter + 1):
            lc_base = self.valutatore.codelength(self.partizione)
            spostamenti = 0
            ordine = self.rng.permutation(len(self.partizione))

            for nodo in tqdm(ordine, desc=f"iter {it} (alpha={self.alpha})", unit="nodo"):
                nodo = int(nodo)
                com_attuale = int(self.partizione[nodo])
                vicini = list(self.grafo.neighbors(nodo))
                if not vicini:
                    continue
                candidate = {int(self.partizione[v]) for v in vicini} - {com_attuale}
                if not candidate:
                    continue

                # perplessita' nella community attuale: calcolata UNA volta per nodo
                h_attuale = self._entropia_sem(nodo, self.membri[com_attuale])
                if h_attuale is None:
                    continue

                miglior_guadagno, miglior_com = self.soglia, None
                for c in candidate:
                    esito = self._valuta_mossa(nodo, c, h_attuale, lc_base)
                    if esito is None:
                        continue
                    guadagno, _, _ = esito
                    if guadagno > miglior_guadagno:
                        miglior_guadagno, miglior_com = guadagno, c

                if miglior_com is not None:
                    self._sposta(nodo, miglior_com)
                    spostamenti += 1
                    lc_base = self.valutatore.codelength(self.partizione)

            lc_fine = self.valutatore.codelength(self.partizione)
            n_com = len(set(self.partizione))
            print(f"\niter {it}: spostamenti={spostamenti} | community={n_com} | L(C)={lc_fine:.4f}")
            storico.append({"iter": it, "spostamenti": spostamenti,
                            "n_community": int(n_com), "codelength": float(lc_fine)})
            np.save(f"{self.dir_salvataggio}/ibrido_alpha{self.alpha}_iter{it}.npy",
                    self.partizione)
            if spostamenti == 0:
                print("nessuno spostamento: convergenza")
                break

        np.save(f"{self.dir_salvataggio}/ibrido_alpha{self.alpha}_finale.npy", self.partizione)
        return self.partizione, storico

In [ ]:
lm = LocalMovingIbrido(
    scorer, val, G, partizione_infomap,
    alpha=0.9,
    sigma_sem=scale["sigma_sem"],
    sigma_lc=scale["sigma_lc"],
    max_iter=4,
)
partizione_ibrida, storico = lm.esegui()

In [ ]:
import numpy as np
import json
from tqdm import tqdm
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
import networkx as nx



def entropia_semantica(scorer, partizione, mostra_avanzamento=True):
    """
    H_sem(C) = media dei log2(PPL) dei nodi, dato il contesto dei co-membri.
    E' l'entropia semantica in bit: la grandezza direttamente confrontabile
    con la description length L(C). Restituisce anche la PPL media (per
    continuita' con i risultati precedenti).
    """
    membri = {}
    for nodo, c in enumerate(partizione):
        membri.setdefault(int(c), []).append(nodo)

    log_ppl, ppl_vals = [], []
    iteratore = range(len(partizione))
    if mostra_avanzamento:
        iteratore = tqdm(iteratore, desc="H_sem(C)", unit="nodo")

    for nodo in iteratore:
        c = int(partizione[nodo])
        ppl = scorer.perplexity_nodo(nodo, membri[c])
        if ppl is not None and ppl > 0:
            ppl_vals.append(ppl)
            log_ppl.append(np.log2(ppl))

    if not log_ppl:
        return float("inf"), float("inf")
    return float(np.mean(log_ppl)), float(np.mean(ppl_vals))


def valuta_partizione_ibrida(scorer, valutatore, partizione, etichette,
                             grafo, alpha, nome="ibrido",
                             file_json="/kaggle/working/risultati_ibrido.json"):
    """
    Valutazione completa di una partizione prodotta dal local-moving ibrido:
    H_sem(C), PPL(C), L(C), NMI, ARI, modularity, e l'obiettivo ibrido.
    Salva/aggiorna un JSON cumulativo, cosi' i risultati dei diversi alpha
    si accumulano nello stesso file.
    """
    h_sem, ppl_media = entropia_semantica(scorer, partizione)
    lc = valutatore.codelength(partizione)
    n_com = len(set(partizione))

    nmi = normalized_mutual_info_score(etichette, partizione)
    ari = adjusted_rand_score(etichette, partizione)

    com_list = {}
    for nodo, c in enumerate(partizione):
        com_list.setdefault(int(c), set()).add(nodo)
    modularity = nx.algorithms.community.modularity(grafo, list(com_list.values()))

    obiettivo = alpha * h_sem + (1 - alpha) * lc

    print(f"\n=== {nome} (alpha={alpha}) ===")
    print(f"community    : {n_com}")
    print(f"H_sem(C)     : {h_sem:.4f} bit")
    print(f"PPL(C)       : {ppl_media:.4f}")
    print(f"L(C)         : {lc:.4f} bit")
    print(f"L_ibrida(C)  : {obiettivo:.4f} bit")
    print(f"NMI          : {nmi:.4f}")
    print(f"ARI          : {ari:.4f}")
    print(f"modularity   : {modularity:.4f}")

    risultato = {
        "alpha": float(alpha), "n_community": int(n_com),
        "h_sem": h_sem, "ppl_media": ppl_media, "codelength": lc,
        "obiettivo_ibrido": float(obiettivo),
        "nmi": float(nmi), "ari": float(ari), "modularity": float(modularity),
    }

    try:
        with open(file_json) as f:
            tutti = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        tutti = {}
    tutti[f"{nome}_alpha{alpha}"] = risultato
    with open(file_json, "w") as f:
        json.dump(tutti, f, indent=2)
    print(f"\nSalvato in {file_json}")
    return risultato

In [ ]:
ris = valuta_partizione_ibrida(
    scorer, val, partizione_ibrida, etichette, G,
    alpha=0.9, nome="ibrido"
)